# Model Pipeline
This file was created for project 5, in order to compile all transformations, and the optimized model itself.
The pipeline will be exported with model to be used in project 5 & exposed with API.

In [1]:
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import numpy as np
import joblib
import pandas as pd
from lib.modelisation import NormalizationPreprocessingTransformer, OneHotEncoderTransformer, OrdinalEncoderTransformer, StandardScalerTransformer, FeatureEngineeringTransformer

# Chargement du dataframe
df = pd.read_csv("./assets/sirh_unified.csv")

# Calculer le scale_pos_weight sur les données d'entraînement
X = df.drop('a_quitte_l_entreprise', axis=1)
y = df['a_quitte_l_entreprise']

# Conversion des labels en 0/1
y = y.map({'Non': 0, 'Oui': 1}).astype(int)

scale_pos_weight = np.sum(y == 0) / np.sum(y == 1)

# Pipeline complète
clf = Pipeline([
    ('custom_preprocessing', NormalizationPreprocessingTransformer()),            
    ('ordinal', OrdinalEncoderTransformer()),                 
    ('feature_engineering', FeatureEngineeringTransformer()), 
    ('scaler', StandardScalerTransformer(['revenu_mensuel'])),
    ('onehot', OneHotEncoderTransformer()),                  
    ('model', XGBClassifier(
        colsample_bytree=1.0,
        learning_rate=0.05,
        max_depth=2,
        n_estimators=500,
        subsample=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric='auc'
    ))
])

# Entraînement
clf.fit(X, y)

# sauvegarde
obj = {
    "pipeline": clf,
    "features": list(X.columns)  # colonnes brutes utilisées pour l'entraînement
}
joblib.dump(obj, "modelp4.pkl")
print("✅ Pipeline complète sauvegardée !")

✅ Pipeline complète sauvegardée !
